### 01 — Generate Sample Data (Bronze Layer)
**Project:** Adobe People Analytics — Semantic Layer Portfolio  
**Author:** Lani Mateo  
**Purpose:** Generate realistic synthetic People Analytics data for 5,000 employees.  
Simulates a mid-size tech company with hiring ramp-up, attrition, compensation changes, and performance reviews.

**Key design decisions:**
- **SCD2 history:** Each employee change (department transfer, promotion, salary adjustment) creates a new version. Enables historical accuracy without headcount inflation.
- **Pay equity gap:** Female employees at IC4+ have salaries adjusted to 91-97% of male counterparts. Creates a detectable ~5-6% gap for the Compensation & Pay Equity report page.
- **90-day exits:** Attrition simulation starts at day 7 post-hire, enabling early turnover detection.
- **Adobe fiscal year:** December = FY start. All date ranges align to Adobe's fiscal calendar.
- **Dynamic dates:** START_DATE and END_DATE derive from the current date and fiscal year logic. Re-running this notebook generates data through the latest complete fiscal period.

**Tables created:**
| Table | Rows (approx) | Description |
|-------|--------------|-------------|
| bronze.employees_raw | ~8,961 | 5,000 employees with SCD2 history |
| bronze.comp_changes_raw | ~6,700 | Salary changes (merit, promotion) |
| bronze.performance_raw | ~9,800 | Performance reviews by cycle |
| bronze.finance_budget_raw | ~1,170 | Department budget by month |
| bronze.dim_date_raw | ~4,748 | Date spine 2018-2030 |

**Run frequency:** Manually as needed. Do NOT schedule daily — each run costs ~$1.50 in compute.  
**Dependencies:** Cell 1 must run first to install faker and python-dateutil.

In [0]:
# =============================================================================
# NOTEBOOK: 01_generate_sample_data
# PURPOSE:  Generate 100K+ rows of realistic People Analytics data
#           and write to people_analytics.bronze.* Delta tables
# CATALOG:  people_analytics
# SCHEMA:   bronze
# AUTHOR:   Portfolio Project — Adobe People Analytics
# DATE:     2026-04-09
# =============================================================================

# CELL 1 — Install dependencies
import subprocess
subprocess.run(["pip", "install", "faker", "python-dateutil", "--quiet"])
print("Dependencies installed")

In [0]:
# =============================================================================
# CELL 2 — Imports and configuration
# =============================================================================

import random
import uuid
from datetime import date, timedelta, datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql.types import *

fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# Configuration
CATALOG         = "people_analytics"
BRONZE_SCHEMA   = "bronze"
NUM_EMPLOYEES   = 5000

# ── Dynamic date range aligned to Adobe fiscal year (starts December) ────
today = date.today()

if today.month >= 12:
    START_DATE = date(today.year - 5, 12, 1)
else:
    START_DATE = date(today.year - 6, 12, 1)

END_DATE = today
MONTHLY_PERIODS = (END_DATE.year - START_DATE.year) * 12 + (END_DATE.month - START_DATE.month)

In [0]:
# =============================================================================
# CELL 3 — Reference data
# =============================================================================

DEPARTMENTS = [
    {"dept_id": "D001", "dept_name": "Software Engineering",    "business_unit": "Engineering",  "cost_centre": "CC001", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D002", "dept_name": "Platform Engineering",    "business_unit": "Engineering",  "cost_centre": "CC002", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D003", "dept_name": "Data Engineering",        "business_unit": "Engineering",  "cost_centre": "CC003", "location": "Seattle",     "region": "Americas"},
    {"dept_id": "D004", "dept_name": "Product Management",      "business_unit": "Product",      "cost_centre": "CC004", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D005", "dept_name": "Product Design",          "business_unit": "Product",      "cost_centre": "CC005", "location": "New York",    "region": "Americas"},
    {"dept_id": "D006", "dept_name": "Product Analytics",       "business_unit": "Product",      "cost_centre": "CC006", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D007", "dept_name": "People Analytics",        "business_unit": "HR",           "cost_centre": "CC007", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D008", "dept_name": "HR Business Partners",    "business_unit": "HR",           "cost_centre": "CC008", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D009", "dept_name": "Talent Acquisition",      "business_unit": "HR",           "cost_centre": "CC009", "location": "Austin",      "region": "Americas"},
    {"dept_id": "D010", "dept_name": "FP&A",                    "business_unit": "Finance",      "cost_centre": "CC010", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D011", "dept_name": "Accounting",              "business_unit": "Finance",      "cost_centre": "CC011", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D012", "dept_name": "Enterprise Sales",        "business_unit": "Sales",        "cost_centre": "CC012", "location": "New York",    "region": "Americas"},
    {"dept_id": "D013", "dept_name": "SMB Sales",               "business_unit": "Sales",        "cost_centre": "CC013", "location": "Austin",      "region": "Americas"},
    {"dept_id": "D014", "dept_name": "Customer Success",        "business_unit": "Sales",        "cost_centre": "CC014", "location": "London",      "region": "EMEA"},
    {"dept_id": "D015", "dept_name": "Digital Marketing",       "business_unit": "Marketing",    "cost_centre": "CC015", "location": "New York",    "region": "Americas"},
    {"dept_id": "D016", "dept_name": "Brand & Content",         "business_unit": "Marketing",    "cost_centre": "CC016", "location": "San Jose",    "region": "Americas"},
    {"dept_id": "D017", "dept_name": "APAC Engineering",        "business_unit": "Engineering",  "cost_centre": "CC017", "location": "Singapore",   "region": "APAC"},
    {"dept_id": "D018", "dept_name": "EMEA Sales",              "business_unit": "Sales",        "cost_centre": "CC018", "location": "London",      "region": "EMEA"},
]

JOB_FAMILIES = {
    "Engineering": [
        {"job_code": "ENG-IC1", "job_title": "Software Engineer I",        "job_level": "IC1", "is_manager": False, "sal_min": 85000,  "sal_max": 115000},
        {"job_code": "ENG-IC2", "job_title": "Software Engineer II",       "job_level": "IC2", "is_manager": False, "sal_min": 110000, "sal_max": 145000},
        {"job_code": "ENG-IC3", "job_title": "Senior Software Engineer",   "job_level": "IC3", "is_manager": False, "sal_min": 140000, "sal_max": 185000},
        {"job_code": "ENG-IC4", "job_title": "Staff Engineer",             "job_level": "IC4", "is_manager": False, "sal_min": 180000, "sal_max": 230000},
        {"job_code": "ENG-IC5", "job_title": "Principal Engineer",         "job_level": "IC5", "is_manager": False, "sal_min": 220000, "sal_max": 300000},
        {"job_code": "ENG-M1",  "job_title": "Engineering Manager",        "job_level": "M1",  "is_manager": True,  "sal_min": 190000, "sal_max": 260000},
        {"job_code": "ENG-M2",  "job_title": "Senior Engineering Manager", "job_level": "M2",  "is_manager": True,  "sal_min": 240000, "sal_max": 320000},
    ],
    "Product": [
        {"job_code": "PM-IC1",  "job_title": "Associate Product Manager",  "job_level": "IC1", "is_manager": False, "sal_min": 80000,  "sal_max": 110000},
        {"job_code": "PM-IC2",  "job_title": "Product Manager",            "job_level": "IC2", "is_manager": False, "sal_min": 105000, "sal_max": 140000},
        {"job_code": "PM-IC3",  "job_title": "Senior Product Manager",     "job_level": "IC3", "is_manager": False, "sal_min": 135000, "sal_max": 180000},
        {"job_code": "PM-IC4",  "job_title": "Principal Product Manager",  "job_level": "IC4", "is_manager": False, "sal_min": 170000, "sal_max": 220000},
        {"job_code": "PM-M1",   "job_title": "Director of Product",        "job_level": "M1",  "is_manager": True,  "sal_min": 185000, "sal_max": 250000},
    ],
    "Data": [
        {"job_code": "DA-IC1",  "job_title": "Data Analyst I",             "job_level": "IC1", "is_manager": False, "sal_min": 75000,  "sal_max": 100000},
        {"job_code": "DA-IC2",  "job_title": "Data Analyst II",            "job_level": "IC2", "is_manager": False, "sal_min": 95000,  "sal_max": 130000},
        {"job_code": "DA-IC3",  "job_title": "Senior Data Analyst",        "job_level": "IC3", "is_manager": False, "sal_min": 125000, "sal_max": 165000},
        {"job_code": "DA-IC4",  "job_title": "Staff Data Scientist",       "job_level": "IC4", "is_manager": False, "sal_min": 160000, "sal_max": 210000},
        {"job_code": "DA-M1",   "job_title": "Analytics Manager",          "job_level": "M1",  "is_manager": True,  "sal_min": 170000, "sal_max": 230000},
    ],
    "Business": [
        {"job_code": "BIZ-IC1", "job_title": "Associate",                  "job_level": "IC1", "is_manager": False, "sal_min": 65000,  "sal_max": 90000},
        {"job_code": "BIZ-IC2", "job_title": "Specialist",                 "job_level": "IC2", "is_manager": False, "sal_min": 85000,  "sal_max": 115000},
        {"job_code": "BIZ-IC3", "job_title": "Senior Specialist",          "job_level": "IC3", "is_manager": False, "sal_min": 110000, "sal_max": 150000},
        {"job_code": "BIZ-M1",  "job_title": "Manager",                    "job_level": "M1",  "is_manager": True,  "sal_min": 130000, "sal_max": 180000},
        {"job_code": "BIZ-M2",  "job_title": "Senior Manager",             "job_level": "M2",  "is_manager": True,  "sal_min": 160000, "sal_max": 210000},
        {"job_code": "BIZ-M3",  "job_title": "Director",                   "job_level": "M3",  "is_manager": True,  "sal_min": 200000, "sal_max": 280000},
    ],
}

COMP_BANDS = [
    {"band_name": "Band 1 (IC1)", "band_min": 65000,  "band_mid": 90000,  "band_max": 115000},
    {"band_name": "Band 2 (IC2)", "band_min": 85000,  "band_mid": 115000, "band_max": 150000},
    {"band_name": "Band 3 (IC3)", "band_min": 110000, "band_mid": 150000, "band_max": 190000},
    {"band_name": "Band 4 (IC4)", "band_min": 150000, "band_mid": 195000, "band_max": 240000},
    {"band_name": "Band 5 (IC5)", "band_min": 200000, "band_mid": 255000, "band_max": 310000},
    {"band_name": "Band 6 (M1)",  "band_min": 170000, "band_mid": 220000, "band_max": 270000},
    {"band_name": "Band 7 (M2+)", "band_min": 220000, "band_mid": 285000, "band_max": 350000},
]

GENDERS      = ["Male", "Female", "Non-binary", "Prefer not to say"]
GENDER_WGTS  = [0.50,   0.44,     0.03,         0.03]

ETHNICITIES  = ["White", "Asian", "Hispanic/Latino", "Black/African American", "Two or more races", "Prefer not to say"]
ETH_WGTS     = [0.48,    0.28,    0.10,              0.08,                     0.04,                0.02]

TERM_REASONS = ["Growth Opportunity", "Compensation", "Manager/Culture", "Life Event", "Involuntary - PIP", "Involuntary - Restructure", "Retirement"]
TERM_WGTS    = [0.38,                  0.20,           0.15,              0.12,         0.05,                0.05,                        0.05]

EMP_TYPES    = ["Full-time", "Part-time", "Contractor"]
EMP_WGTS     = [0.85,        0.08,        0.07]

print("Reference data loaded:")
print(f"  Departments:  {len(DEPARTMENTS)}")
print(f"  Job families: {len(JOB_FAMILIES)}")
print(f"  Comp bands:   {len(COMP_BANDS)}")

In [0]:
# =============================================================================
# CELL 4 — Helper functions
# =============================================================================

def random_date(start, end):
    delta = (end - start).days
    if delta <= 0:
        return start
    return start + timedelta(days=random.randint(0, delta))

def get_all_jobs():
    jobs = []
    for family, roles in JOB_FAMILIES.items():
        for role in roles:
            jobs.append({**role, "job_family": family})
    return jobs

ALL_JOBS = get_all_jobs()

def get_job_band(job_level):
    mapping = {
        "IC1": "Band 1 (IC1)", "IC2": "Band 2 (IC2)", "IC3": "Band 3 (IC3)",
        "IC4": "Band 4 (IC4)", "IC5": "Band 5 (IC5)", "M1":  "Band 6 (M1)",
        "M2":  "Band 7 (M2+)", "M3":  "Band 7 (M2+)"
    }
    return mapping.get(job_level, "Band 2 (IC2)")

def get_attrition_rate(tenure_years):
    annual_rate = (
        0.08 if tenure_years < 1 else
        0.12 if tenure_years < 3 else
        0.07 if tenure_years < 5 else
        0.03
    )
    return annual_rate / 12

def gender_salary_adjustment(gender, job_level):
    if gender == "Female" and job_level in ["IC4", "IC5", "M1", "M2", "M3"]:
        return random.uniform(0.91, 0.97)
    return 1.0

print("Helper functions defined.")

In [0]:
# =============================================================================
# CELL 5 — Generate employees with SCD2 history
# =============================================================================

print("Generating employee records with SCD2 history...")

employees_raw = []
emp_counter   = 1

for i in range(NUM_EMPLOYEES):
    emp_id     = f"E{emp_counter:06d}"
    emp_counter += 1

    gender     = random.choices(GENDERS,      weights=GENDER_WGTS)[0]
    ethnicity  = random.choices(ETHNICITIES,  weights=ETH_WGTS)[0]
    emp_type   = random.choices(EMP_TYPES,    weights=EMP_WGTS)[0]
    birth_date = random_date(date(1965, 1, 1), date(2000, 12, 31))

    job     = random.choice(ALL_JOBS)
    dept    = random.choice(DEPARTMENTS)
    sal_adj = gender_salary_adjustment(gender, job["job_level"])
    salary  = round(random.uniform(job["sal_min"], job["sal_max"]) * sal_adj, -2)
    band    = get_job_band(job["job_level"])

    # Hire dates spread dynamically across data range with more recent hires
    hire_years = list(range(START_DATE.year, END_DATE.year + 1))
    hire_weights = [1.0 * (1.12 ** i) for i in range(len(hire_years))]
    year = random.choices(hire_years, weights=hire_weights)[0]
    year_start = max(date(year, 1, 1), START_DATE)
    year_end = min(date(year, 12, 31), END_DATE - timedelta(days=30))
    if year_end < year_start:
        year_end = year_start
    hire_date = random_date(year_start, year_end)

    current_dept   = dept
    current_job    = job
    current_sal    = salary
    current_band   = band
    current_status = "Active"
    version        = 1
    terminated     = False

    employees_raw.append({
        "emp_id":             emp_id,
        "version":            version,
        "full_name":          fake.name(),
        "gender":             gender,
        "ethnicity":          ethnicity,
        "birth_date":         birth_date.isoformat(),
        "emp_type":           emp_type,
        "hire_date":          hire_date.isoformat(),
        "dept_id":            current_dept["dept_id"],
        "dept_name":          current_dept["dept_name"],
        "business_unit":      current_dept["business_unit"],
        "location":           current_dept["location"],
        "region":             current_dept["region"],
        "cost_centre":        current_dept["cost_centre"],
        "job_code":           current_job["job_code"],
        "job_title":          current_job["job_title"],
        "job_family":         current_job["job_family"],
        "job_level":          current_job["job_level"],
        "is_people_manager":  current_job["is_manager"],
        "salary":             current_sal,
        "comp_band":          current_band,
        "manager_id":         None,
        "status":             current_status,
        "termination_date":   None,
        "termination_reason": None,
        "scd_start_date":     hire_date.isoformat(),
        "scd_end_date":       None,
        "is_current":         1,
        "load_ts":            datetime.now().isoformat(),
    })

    sim_date = hire_date + timedelta(days=7)

    while sim_date <= END_DATE and not terminated:
        tenure_years = (sim_date - hire_date).days / 365.25

        # Attrition check
        if random.random() < get_attrition_rate(tenure_years):
            term_reason = random.choices(TERM_REASONS, weights=TERM_WGTS)[0]
            employees_raw[-1]["scd_end_date"]      = (sim_date - timedelta(days=1)).isoformat()
            employees_raw[-1]["is_current"]         = 0
            employees_raw[-1]["termination_date"]   = sim_date.isoformat()
            employees_raw[-1]["termination_reason"] = term_reason
            version += 1
            employees_raw.append({
                **employees_raw[-1],
                "version":            version,
                "status":             "Terminated",
                "termination_date":   sim_date.isoformat(),
                "termination_reason": term_reason,
                "scd_start_date":     sim_date.isoformat(),
                "scd_end_date":       None,
                "is_current":         1,
                "load_ts":            datetime.now().isoformat(),
            })
            terminated = True
            break

        # Department transfer ~5% annual
        if random.random() < 0.05 / 12:
            new_dept = random.choice([d for d in DEPARTMENTS if d["dept_id"] != current_dept["dept_id"]])
            employees_raw[-1]["scd_end_date"] = (sim_date - timedelta(days=1)).isoformat()
            employees_raw[-1]["is_current"]   = 0
            current_dept = new_dept
            version += 1
            employees_raw.append({
                **employees_raw[-1],
                "version":       version,
                "dept_id":       current_dept["dept_id"],
                "dept_name":     current_dept["dept_name"],
                "business_unit": current_dept["business_unit"],
                "location":      current_dept["location"],
                "region":        current_dept["region"],
                "cost_centre":   current_dept["cost_centre"],
                "scd_start_date": sim_date.isoformat(),
                "scd_end_date":  None,
                "is_current":    1,
                "load_ts":       datetime.now().isoformat(),
            })

        # Promotion ~8% annual
        elif random.random() < 0.08 / 12:
            level_order = ["IC1","IC2","IC3","IC4","IC5","M1","M2","M3"]
            curr_idx    = level_order.index(current_job["job_level"]) if current_job["job_level"] in level_order else 0
            if curr_idx < len(level_order) - 1:
                next_level = level_order[curr_idx + 1]
                new_jobs   = [j for j in ALL_JOBS if j["job_level"] == next_level and j["job_family"] == current_job["job_family"]]
                if new_jobs:
                    new_job      = random.choice(new_jobs)
                    sal_adj      = gender_salary_adjustment(gender, new_job["job_level"])
                    new_sal      = round(random.uniform(new_job["sal_min"], new_job["sal_max"]) * sal_adj, -2)
                    new_band     = get_job_band(new_job["job_level"])
                    employees_raw[-1]["scd_end_date"] = (sim_date - timedelta(days=1)).isoformat()
                    employees_raw[-1]["is_current"]   = 0
                    current_job  = new_job
                    current_sal  = new_sal
                    current_band = new_band
                    version += 1
                    employees_raw.append({
                        **employees_raw[-1],
                        "version":          version,
                        "job_code":         current_job["job_code"],
                        "job_title":        current_job["job_title"],
                        "job_level":        current_job["job_level"],
                        "is_people_manager": current_job["is_manager"],
                        "salary":           current_sal,
                        "comp_band":        current_band,
                        "scd_start_date":   sim_date.isoformat(),
                        "scd_end_date":     None,
                        "is_current":       1,
                        "load_ts":          datetime.now().isoformat(),
                    })

        # Annual merit raise each January
        elif sim_date.month == 1 and sim_date.day <= 31 and tenure_years > 0.5:
            if random.random() < 0.85:
                merit_pct   = random.uniform(0.02, 0.06)
                new_sal     = round(current_sal * (1 + merit_pct), -2)
                employees_raw[-1]["scd_end_date"] = (sim_date - timedelta(days=1)).isoformat()
                employees_raw[-1]["is_current"]   = 0
                current_sal = new_sal
                version += 1
                employees_raw.append({
                    **employees_raw[-1],
                    "version":        version,
                    "salary":         current_sal,
                    "scd_start_date": sim_date.isoformat(),
                    "scd_end_date":   None,
                    "is_current":     1,
                    "load_ts":        datetime.now().isoformat(),
                })

        sim_date += timedelta(days=random.randint(25, 35))

df_employees = pd.DataFrame(employees_raw)

print(f"Generated {len(df_employees)} employee records")
print(f"  Unique employees:    {df_employees['emp_id'].nunique()}")
print(f"  Current rows:        {len(df_employees[df_employees['is_current']==1])}")
print(f"  SCD2 history rows:   {len(df_employees[df_employees['is_current']==0])}")
print(f"  Terminated:          {len(df_employees[(df_employees['is_current']==1) & (df_employees['status']=='Terminated')])}")


In [0]:
# =============================================================================
# CELL 6 — Assign managers
# =============================================================================

print("Assigning managers...")

active_managers = df_employees[
    (df_employees["is_current"] == 1) &
    (df_employees["is_people_manager"] == True) &
    (df_employees["status"] == "Active")
]["emp_id"].tolist()

for idx, row in df_employees[df_employees["is_current"] == 1].iterrows():
    if row["is_people_manager"] and row["status"] == "Active":
        continue
    if active_managers:
        df_employees.at[idx, "manager_id"] = random.choice(active_managers)

print(f"Managers assigned. Active managers: {len(active_managers)}")

In [0]:
# =============================================================================
# CELL 7 — Generate compensation changes
# =============================================================================

print("Generating compensation change records...")

comp_changes = []
comp_id      = 1

for emp_id in df_employees[df_employees["is_current"] == 1]["emp_id"].unique():
    emp_versions = df_employees[df_employees["emp_id"] == emp_id].sort_values("scd_start_date")
    prev_salary  = None
    for _, ver in emp_versions.iterrows():
        if prev_salary is not None and ver["salary"] != prev_salary:
            change_pct = (ver["salary"] - prev_salary) / prev_salary
            reason     = (
                "Promotion"        if change_pct > 0.10 else
                "Merit"            if change_pct > 0.02 else
                "Adjustment"       if change_pct < 0    else
                "Equity Adjustment"
            )
            band_info   = next((b for b in COMP_BANDS if b["band_name"] == ver["comp_band"]), COMP_BANDS[1])
            compa_ratio = round(ver["salary"] / band_info["band_mid"], 3)
            range_pen   = round(
                (ver["salary"] - band_info["band_min"]) /
                max(band_info["band_max"] - band_info["band_min"], 1), 3
            )
            comp_changes.append({
                "comp_change_id":    f"CC{comp_id:07d}",
                "emp_id":            emp_id,
                "effective_date":    ver["scd_start_date"],
                "old_salary":        prev_salary,
                "new_salary":        ver["salary"],
                "change_amount":     round(ver["salary"] - prev_salary, 2),
                "change_pct":        round(change_pct * 100, 2),
                "change_reason":     reason,
                "comp_band":         ver["comp_band"],
                "compa_ratio":       compa_ratio,
                "range_penetration": max(0, min(1, range_pen)),
                "load_ts":           datetime.now().isoformat(),
            })
            comp_id += 1
        prev_salary = ver["salary"]

df_comp = pd.DataFrame(comp_changes)
print(f"Generated {len(df_comp)} compensation change records")

In [0]:
# =============================================================================
# CELL 8 — Generate performance reviews
# =============================================================================

print("Generating performance review records...")

# ── Dynamic review cycles based on data range ────────────────────────────────
REVIEW_CYCLES = []
REVIEW_DATES  = {}

for y in range(START_DATE.year, END_DATE.year + 1):
    h1_date = date(y, 6, 30)
    h2_date = date(y, 12, 31)
    if h1_date >= START_DATE and h1_date <= END_DATE:
        cycle = f"{y}H1"
        REVIEW_CYCLES.append(cycle)
        REVIEW_DATES[cycle] = h1_date
    if h2_date >= START_DATE and h2_date <= END_DATE:
        cycle = f"{y}H2"
        REVIEW_CYCLES.append(cycle)
        REVIEW_DATES[cycle] = h2_date

print(f"  Review cycles: {REVIEW_CYCLES}")

RATING_WEIGHTS = [0.05, 0.15, 0.50, 0.25, 0.05]
RATING_LABELS  = {
    1: "Needs Development",
    2: "Approaching",
    3: "Meets Expectations",
    4: "Exceeds Expectations",
    5: "Outstanding"
}

reviews   = []
review_id = 1

for _, emp in df_employees[df_employees["is_current"] == 1].iterrows():
    hire_dt = date.fromisoformat(str(emp["hire_date"])[:10])
    for cycle in REVIEW_CYCLES:
        review_dt = REVIEW_DATES[cycle]
        if hire_dt > review_dt:
            continue
        if emp["status"] == "Terminated" and emp["termination_date"] is not None:
            term_dt = date.fromisoformat(str(emp["termination_date"])[:10])
            if term_dt < review_dt:
                continue
        rating = random.choices([1, 2, 3, 4, 5], weights=RATING_WEIGHTS)[0]
        reviews.append({
            "review_id":         f"R{review_id:08d}",
            "emp_id":            emp["emp_id"],
            "review_cycle":      cycle,
            "review_date":       review_dt.isoformat(),
            "rating":            rating,
            "rating_label":      RATING_LABELS[rating],
            "is_high_performer": 1 if rating >= 4 else 0,
            "is_pip":            1 if rating == 1 else 0,
            "manager_id":        emp["manager_id"],
            "dept_id":           emp["dept_id"],
            "load_ts":           datetime.now().isoformat(),
        })
        review_id += 1

df_reviews = pd.DataFrame(reviews)
print(f"Generated {len(df_reviews)} performance review records")

In [0]:
# =============================================================================
# CELL 9 — Generate finance headcount budget
# =============================================================================

print("Generating finance budget records...")

# Calculate actual headcount per department per month from snapshots
# Budget should track close to actual with small variance
budget_rows = []
budget_id   = 1

for dept in DEPARTMENTS:
    # Get this department's actual HC from the employee data
    dept_employees = df_employees[
        (df_employees['dept_id'] == dept['dept_id']) &
        (df_employees['is_current'] == 1) &
        (df_employees['status'] == 'Active')
    ]
    final_hc = len(dept_employees)
    
    for month_offset in range(MONTHLY_PERIODS):
        budget_date = START_DATE + relativedelta(months=month_offset)
        
        # Budget grows linearly from ~60% of final HC to ~105% of final HC
        # Mimics a realistic hiring plan that slightly outpaces actual hiring
        progress = month_offset / max(MONTHLY_PERIODS - 1, 1)
        planned_hc = round(final_hc * (0.85 + progress * 0.20) * random.uniform(0.97, 1.03))
        planned_hc = max(planned_hc, 5)
        
        avg_salary = random.uniform(110000, 160000)
        
        if budget_date.month >= 12:
            fiscal_yr = budget_date.year + 1
        else:
            fiscal_yr = budget_date.year
            
        budget_rows.append({
            "budget_id":          f"B{budget_id:07d}",
            "dept_id":            dept["dept_id"],
            "dept_name":          dept["dept_name"],
            "business_unit":      dept["business_unit"],
            "budget_month":       budget_date.isoformat(),
            "budget_year":        budget_date.year,
            "budget_month_num":   budget_date.month,
            "fiscal_year":        fiscal_yr,
            "approved_headcount": planned_hc,
            "approved_payroll":   round(planned_hc * avg_salary / 12, 2),
            "avg_payroll_per_hc": round(avg_salary / 12, 2),
            "load_ts":            datetime.now().isoformat(),
        })
        budget_id += 1

df_budget = pd.DataFrame(budget_rows)
print(f"Generated {len(df_budget)} finance budget records")
print(f"  Total approved HC (latest month): {sum(r['approved_headcount'] for r in budget_rows[-18:])}")
print(f"  Total actual HC: {len(df_employees[(df_employees['is_current']==1) & (df_employees['status']=='Active')])}")

In [0]:
# =============================================================================
# CELL 10 — Generate dim_date spine (dynamic, Adobe fiscal year starts December)
# =============================================================================
print("Generating date spine...")

# Spine extends 2 years before START_DATE and 4 years after END_DATE
# to support time intelligence lookback and forward planning
spine_start = date(START_DATE.year - 2, 1, 1)
spine_end   = date(END_DATE.year + 4, 12, 31)

date_rows = []
current   = spine_start

while current <= spine_end:
    # Adobe fiscal year: December belongs to NEXT fiscal year
    if current.month >= 12:
        fiscal_year = current.year + 1
    else:
        fiscal_year = current.year

    # Fiscal quarter: Dec-Feb = FQ1, Mar-May = FQ2, Jun-Aug = FQ3, Sep-Nov = FQ4
    if current.month in (12, 1, 2):
        fiscal_quarter = 1
    elif current.month in (3, 4, 5):
        fiscal_quarter = 2
    elif current.month in (6, 7, 8):
        fiscal_quarter = 3
    else:
        fiscal_quarter = 4

    # Fiscal month: Dec = 1, Jan = 2, ..., Nov = 12
    if current.month == 12:
        fiscal_month = 1
    else:
        fiscal_month = current.month + 1

    date_rows.append({
        "date_sk":        int(current.strftime("%Y%m%d")),
        "date":           current.isoformat(),
        "year":           current.year,
        "quarter":        (current.month - 1) // 3 + 1,
        "month":          current.month,
        "month_name":     current.strftime("%B"),
        "month_short":    current.strftime("%b"),
        "week":           current.isocalendar()[1],
        "day":            current.day,
        "day_of_week":    current.weekday() + 1,
        "day_name":       current.strftime("%A"),
        "is_weekday":     1 if current.weekday() < 5 else 0,
        "is_month_end":   1 if (current + timedelta(days=1)).month != current.month else 0,
        "is_quarter_end": 1 if current.month in [3, 6, 9, 12] and (current + timedelta(days=1)).month != current.month else 0,
        "fiscal_year":    fiscal_year,
        "fiscal_quarter": fiscal_quarter,
        "fiscal_month":   fiscal_month,
        "yyyymm":         int(current.strftime("%Y%m")),
    })
    current += timedelta(days=1)

df_dates = pd.DataFrame(date_rows)
print(f"Generated {len(df_dates)} date spine records ({spine_start} to {spine_end})")
print(f"  Adobe fiscal year mapping: Dec {END_DATE.year - 1} = FY{END_DATE.year}")
print(f"  Fiscal quarters: FQ1=Dec-Feb, FQ2=Mar-May, FQ3=Jun-Aug, FQ4=Sep-Nov")

In [0]:
# =============================================================================
# CELL 11 — Write all tables to Delta Lake
# =============================================================================

print("\nWriting tables to Databricks Delta Lake...")
print(f"Target: {CATALOG}.{BRONZE_SCHEMA}")

spark = SparkSession.builder.getOrCreate()
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

def write_table(df_pandas, table_name, partition_col=None):
    print(f"\n  Writing {table_name}...")
    for col in df_pandas.columns:
        if "date" in col.lower() and df_pandas[col].dtype == object:
            df_pandas[col] = pd.to_datetime(df_pandas[col], errors="coerce")
    df_spark = spark.createDataFrame(df_pandas)
    writer   = df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_col and partition_col in df_pandas.columns:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}").count()
    print(f"  ✓ {table_name}: {count:,} rows written")
    return count

counts = {}
counts["employees_raw"]      = write_table(df_employees, "employees_raw",      partition_col="status")
counts["comp_changes_raw"]   = write_table(df_comp,      "comp_changes_raw")
counts["performance_raw"]    = write_table(df_reviews,   "performance_raw")
counts["finance_budget_raw"] = write_table(df_budget,    "finance_budget_raw")
counts["dim_date_raw"]       = write_table(df_dates,     "dim_date_raw")

total = sum(counts.values())
print(f"\n{'='*55}")
print(f"DATA GENERATION COMPLETE")
print(f"{'='*55}")
for tbl, cnt in counts.items():
    print(f"  {CATALOG}.{BRONZE_SCHEMA}.{tbl:<25} {cnt:>8,} rows")
print(f"{'='*55}")
print(f"  TOTAL ROWS:                              {total:>8,}")
print(f"{'='*55}")

In [0]:
# =============================================================================
# CELL 12 — Verify
# =============================================================================

print("\nVerifying tables...")
spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}").show()

print("\nSample from employees_raw (SCD2 view):")
spark.sql(f"""
    SELECT emp_id, full_name, dept_name, job_title, salary, status,
           scd_start_date, scd_end_date, is_current, version
    FROM {CATALOG}.{BRONZE_SCHEMA}.employees_raw
    ORDER BY emp_id, version
    LIMIT 10
""").show(truncate=False)

print("\nSCD2 summary:")
spark.sql(f"""
    SELECT status, is_current,
           COUNT(*)              AS record_count,
           COUNT(DISTINCT emp_id) AS unique_employees
    FROM {CATALOG}.{BRONZE_SCHEMA}.employees_raw
    GROUP BY status, is_current
    ORDER BY status, is_current
""").show()

print("\nDate range check:")
spark.sql(f"""
    SELECT
        MIN(hire_date) AS earliest_hire,
        MAX(hire_date) AS latest_hire,
        COUNT(DISTINCT emp_id) AS total_employees
    FROM {CATALOG}.{BRONZE_SCHEMA}.employees_raw
    WHERE is_current = 1
""").show()

print("\nAll done! Bronze layer populated — ready for Silver transformation.")